# Fine-tune T5 on Twitter Customer Service Data
### Goal: train T5 to generate support replies given a customer message
---

## Step 1: Install required libraries

In [1]:
!pip install transformers datasets accelerate sentencepiece rouge-score -q

  Preparing metadata (setup.py) ... done


## Step 2: Import libraries

In [2]:
import pandas as pd
import numpy as np
import torch
import re
from datasets import Dataset, DatasetDict
from transformers import (
    T5Tokenizer,
    T5ForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback
)
from rouge_score import rouge_scorer
import warnings
warnings.filterwarnings('ignore')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)
if device == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
    print('GPU Memory:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

Device: cpu


## Step 3: Load the dataset
Source: **twcs** (Twitter Customer Support) - real conversations between customers and support teams from companies like Apple, Amazon, Uber.

In [3]:
url = "https://raw.githubusercontent.com/paulperry/twitterCS/master/twcs.csv"
try:
    df = pd.read_csv(url)
    print('Loaded from URL')
except:

    print('URL failed. Using sample data.')
    data = {
        'tweet_id': range(1, 21),
        'author_id': ['customer'] * 10 + ['AppleSupport'] * 10,
        'inbound': [True] * 10 + [False] * 10,
        'created_at': ['Mon Oct 31 22:10:47 +0000 2017'] * 20,
        'text': [
            '@AppleSupport My iPhone will not charge. I tried different cables.',
            '@AmazonHelp My package shows delivered but I never received it.',
            '@Uber_Support I was charged twice for the same ride.',
            '@Delta My flight was cancelled and I need a refund.',
            '@SpotifyCares The app keeps crashing every time I open it.',
            '@AppleSupport Face ID stopped working after the iOS update.',
            '@AmazonHelp I received a damaged item.',
            '@Uber_Support Driver never showed up but I was charged a fee.',
            '@Delta I need to change my seat on flight DL123.',
            '@SpotifyCares Cannot download songs offline. I am on Premium.',
            'Sorry about that. Try a force restart: Volume Up, Volume Down, hold Side button.',
            'Sorry to hear that. DM us your order number and we will reship it.',
            'Sorry for the double charge. DM your trip details and we will refund within 3-5 days.',
            'Sorry about your cancelled flight. DM your booking reference for refund help.',
            'Sorry about that. Try reinstalling the app. DM us if the issue continues.',
            'Go to Settings, then Face ID, then Reset Face ID to fix it.',
            'Sorry. DM your order ID and damage photos and we will send a replacement.',
            'The cancellation fee will be waived. DM your trip ID for processing.',
            'Go to My Trips, select your flight, then Manage Seats.',
            'Try logging out and back in. Check your storage space. DM us if it continues.'
        ],
        'response_tweet_id': [str(i + 10) for i in range(10)] + [None] * 10,
        'in_response_to_tweet_id': [None] * 10 + [str(i + 1) for i in range(10)]
    }
    df = pd.DataFrame(data)

print('Shape   :', df.shape)
print('Columns :', list(df.columns))
df.head(3)

URL failed. Using sample data.
Shape   : (20, 7)
Columns : ['tweet_id', 'author_id', 'inbound', 'created_at', 'text', 'response_tweet_id', 'in_response_to_tweet_id']


,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
0,1,customer,True,Mon Oct 31 22:10:47 +0000 2017,@AppleSupport My iPhone will not charge. I tri...,10,None
1,2,customer,True,Mon Oct 31 22:10:47 +0000 2017,@AmazonHelp My package shows delivered but I n...,11,None
2,3,customer,True,Mon Oct 31 22:10:47 +0000 2017,@Uber_Support I was charged twice for the same...,12,None


In [4]:
# basic data exploration
print('Total rows:', len(df))
print('\nMissing values:')
print(df.isnull().sum())
print('\nColumn types:')
print(df.dtypes)

Total rows: 20

Missing values:
tweet_id                    0
author_id                   0
inbound                     0
created_at                  0
text                        0
response_tweet_id          10
in_response_to_tweet_id    10
dtype: int64

Column types:
tweet_id                    int64
author_id                  object
inbound                      bool
created_at                 object
text                       object
response_tweet_id          object
in_response_to_tweet_id    object
dtype: object


## Step 4: Build input-output pairs (customer message -> support reply)

In [5]:
def clean_text(text):
    # remove @mentions
    text = re.sub(r'@\w+', '', str(text))
    # remove URLs
    text = re.sub(r'http\S+|www\.\S+', '', text)
    # remove agent signature codes like ^AB
    text = re.sub(r'\^[A-Z]{2,3}', '', text)
    # remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def build_qa_pairs(df):
    pairs = []

    if 'inbound' in df.columns:
        # separate customer messages from support replies
        customer_msgs = df[df['inbound'] == True].copy()
        support_msgs  = df[df['inbound'] == False].copy()

        # build a lookup: tweet_id -> reply text
        support_dict = {}
        if 'in_response_to_tweet_id' in df.columns:
            for _, row in support_msgs.iterrows():
                if pd.notna(row.get('in_response_to_tweet_id')):
                    support_dict[str(row['in_response_to_tweet_id'])] = row['text']

        for _, row in customer_msgs.iterrows():
            tweet_id      = str(row['tweet_id'])
            customer_text = clean_text(row['text'])

            # find the reply for this customer message
            response_text = None
            if tweet_id in support_dict:
                response_text = clean_text(support_dict[tweet_id])
            elif 'response_tweet_id' in df.columns and pd.notna(row.get('response_tweet_id')):
                resp_id = str(int(float(row['response_tweet_id'])))
                if resp_id in support_dict:
                    response_text = clean_text(support_dict[resp_id])

            # skip pairs where either side is too short
            if customer_text and response_text and len(customer_text) > 10 and len(response_text) > 10:
                pairs.append({
                    'input_text' : f'customer support: {customer_text}',
                    'target_text': response_text
                })
    else:
        # fallback: treat odd rows as questions and even rows as answers
        for i in range(0, len(df) - 1, 2):
            q = clean_text(df.iloc[i]['text'])
            a = clean_text(df.iloc[i + 1]['text'])
            if q and a:
                pairs.append({'input_text': f'customer support: {q}', 'target_text': a})

    return pd.DataFrame(pairs)


qa_df = build_qa_pairs(df)
print('Total pairs:', len(qa_df))

for i, row in qa_df.head(3).iterrows():
    print('\n--- Example', i + 1, '---')
    print('Input :', row['input_text'][:100])
    print('Target:', row['target_text'][:100])

Total pairs: 10

--- Example 1 ---
Input : customer support: My iPhone will not charge. I tried different cables.
Target: Sorry about that. Try a force restart: Volume Up, Volume Down, hold Side button.

--- Example 2 ---
Input : customer support: My package shows delivered but I never received it.
Target: Sorry to hear that. DM us your order number and we will reship it.

--- Example 3 ---
Input : customer support: I was charged twice for the same ride.
Target: Sorry for the double charge. DM your trip details and we will refund within 3-5 days.


In [6]:
# remove pairs that are too short, too long, or duplicated
qa_df = qa_df[
    (qa_df['input_text'].str.len() >= 20) &
    (qa_df['input_text'].str.len() <= 512) &
    (qa_df['target_text'].str.len() >= 10) &
    (qa_df['target_text'].str.len() <= 512)
].drop_duplicates().reset_index(drop=True)

print('Pairs after filtering:', len(qa_df))
print('Avg input length :', qa_df['input_text'].str.len().mean())
print('Avg target length:', qa_df['target_text'].str.len().mean())

Pairs after filtering: 10
Avg input length : 61.5
Avg target length: 71.2


## Step 5: Split into train, validation, and test sets

In [7]:
from sklearn.model_selection import train_test_split

# 80% train, 10% validation, 10% test
train_df, temp_df = train_test_split(qa_df, test_size=0.2, random_state=42)
val_df, test_df   = train_test_split(temp_df, test_size=0.5, random_state=42)

print('Train :', len(train_df))
print('Val   :', len(val_df))
print('Test  :', len(test_df))

# convert to HuggingFace DatasetDict
dataset = DatasetDict({
    'train'     : Dataset.from_pandas(train_df.reset_index(drop=True)),
    'validation': Dataset.from_pandas(val_df.reset_index(drop=True)),
    'test'      : Dataset.from_pandas(test_df.reset_index(drop=True))
})
print(dataset)

Train : 8
Val   : 1
Test  : 1
DatasetDict({
    train: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 8
    })
    validation: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 1
    })
    test: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 1
    })
})


## Step 6: Load T5 model and tokenizer

In [8]:
# choose model size based on your hardware:
# t5-small -> fast, good for testing (60M params)
# t5-base  -> balanced, recommended (220M params)
# t5-large -> best quality, needs a strong GPU (770M params)

MODEL_NAME = 't5-small'

tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)
model     = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)
model     = model.to(device)

num_params = sum(p.numel() for p in model.parameters()) / 1e6
print('Model      :', MODEL_NAME)
print('Parameters :', round(num_params, 1), 'M')

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model      : t5-small
Parameters : 60.5 M


## Step 7: Tokenize the dataset

In [10]:
MAX_INPUT_LEN  = 128
MAX_TARGET_LEN = 128


def tokenize_function(examples):
    # tokenize the input (customer message)
    model_inputs = tokenizer(
        examples['input_text'],
        max_length=MAX_INPUT_LEN,
        padding='max_length',
        truncation=True
    )

    # tokenize the target (support reply)
    labels = tokenizer(
        text_target=examples['target_text'],
        max_length=MAX_TARGET_LEN,
        padding='max_length',
        truncation=True
    )


    label_ids = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label]
        for label in labels['input_ids']
    ]

    model_inputs['labels'] = label_ids
    return model_inputs


tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=['input_text', 'target_text'],
    desc='Tokenizing'
)

print('Done')
print(tokenized_dataset)

Tokenizing:   0%|          | 0/8 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/1 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/1 [00:00<?, ? examples/s]

Done
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 8
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 1
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 1
    })
})


## Step 8: Set training arguments

In [12]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100,
    pad_to_multiple_of=8
)

training_args = Seq2SeqTrainingArguments(
    output_dir='./t5-twitter-customer-service',

    # number of full passes over the training data
    num_train_epochs=3,

    # samples processed per step per device (reduce to 4 on CPU)
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    # accumulate gradients over N steps before updating weights
    gradient_accumulation_steps=2,

    # optimizer settings
    learning_rate=5e-4,
    warmup_ratio=0.1,
    weight_decay=0.01,
    lr_scheduler_type='cosine',

    # evaluation_strategy was renamed to eval_strategy in newer transformers
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,

    # run model.generate during evaluation to compute ROUGE
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LEN,

    logging_dir='./logs',
    logging_steps=50,
    report_to='none',

    # fp16 mixed precision speeds up training on GPU
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=2,
    save_total_limit=2,
)

effective_batch = training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps
print('Effective batch size:', effective_batch)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Effective batch size: 16


## Step 9: Define evaluation metric (ROUGE)

In [13]:
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)


def compute_metrics(eval_preds):
    predictions, labels = eval_preds

    # decode predicted token ids back to text
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)

    # restore -100 positions to pad token id before decoding labels
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds  = [p.strip() for p in decoded_preds]
    decoded_labels = [l.strip() for l in decoded_labels]

    # compute ROUGE for each sample then return the average
    all_scores = {'rouge1': [], 'rouge2': [], 'rougeL': []}
    for pred, label in zip(decoded_preds, decoded_labels):
        if pred and label:
            scores = scorer.score(label, pred)
            for key in all_scores:
                all_scores[key].append(scores[key].fmeasure)

    return {
        'rouge1': np.mean(all_scores['rouge1']),
        'rouge2': np.mean(all_scores['rouge2']),
        'rougeL': np.mean(all_scores['rougeL']),
    }

## Step 10: Train the model

In [16]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['validation'],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    # stop early if validation loss does not improve for 3 epochs
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

train_result = trainer.train()

print('Training time (s) :', train_result.metrics['train_runtime'])
print('Final train loss   :', train_result.metrics['train_loss'])

# save the best model checkpoint
trainer.save_model('./t5-twitter-customer-service/best_model')
tokenizer.save_pretrained('./t5-twitter-customer-service/best_model')
print('Model saved.')

Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel
1,No log,4.694294,0.380952,0.000000,0.285714
2,No log,4.762042,0.421053,0.000000,0.315789
3,No log,4.480946,0.421053,0.000000,0.315789


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


Training time (s) : 80.375
Final train loss   : 5.343566258748372


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved.


## Step 11: Evaluate on the test set

In [17]:
test_results = trainer.evaluate(tokenized_dataset['test'])

print('Test set results:')
for key, val in test_results.items():
    if isinstance(val, float):
        print(f'  {key}: {val:.4f}')

Test set results:
  eval_loss: 3.8409
  eval_rouge1: 0.0000
  eval_rouge2: 0.0000
  eval_rougeL: 0.0000
  eval_runtime: 1.1391
  eval_samples_per_second: 0.8780
  eval_steps_per_second: 0.8780
  epoch: 3.0000


## Step 12: Generate responses (inference)

In [18]:
def generate_response(customer_message, model, tokenizer, device, max_length=128, num_beams=4):
    # add the task prefix that T5 was fine-tuned with
    input_text = f'customer support: {customer_message}'

    inputs = tokenizer(
        input_text,
        return_tensors='pt',
        max_length=128,
        truncation=True
    ).to(device)

    model.eval()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=max_length,
            num_beams=num_beams,        # beam search produces better text than greedy
            no_repeat_ngram_size=3,     # prevent the model from repeating the same phrases
            early_stopping=True,
            length_penalty=1.5,         # favor longer, more complete replies
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)


# run on a few example messages to check quality
test_messages = [
    "My iPhone won't turn on after the latest update.",
    "My order was supposed to arrive 3 days ago but I still haven't received it.",
    "I was charged twice for the same purchase.",
    "The app keeps crashing every time I open it.",
    "I need to cancel my subscription but can't find the option."
]

print('-' * 60)
for i, msg in enumerate(test_messages, 1):
    response = generate_response(msg, model, tokenizer, device)
    print(f'[{i}] Customer : {msg}')
    print(f'    Response: {response}')
    print('-' * 60)

------------------------------------------------------------
[1] Customer : My iPhone won't turn on after the latest update.
    Response: My iPhone won't turn on after the latest update.
------------------------------------------------------------
[2] Customer : My order was supposed to arrive 3 days ago but I still haven't received it.
    Response: My order was supposed to arrive 3 days ago but I still haven't received it.
------------------------------------------------------------
[3] Customer : I was charged twice for the same purchase.
    Response: Ich wurde doppelt für denselben Kauf berechnet.
------------------------------------------------------------
[4] Customer : The app keeps crashing every time I open it.
    Response: Customer Service: The app keeps crashing every time I open it.
------------------------------------------------------------
[5] Customer : I need to cancel my subscription but can't find the option.
    Response: I need to cancel my subscription, but can

## Step 13: Compare generated replies with ground truth

In [19]:
n_samples    = min(5, len(test_df))
test_samples = test_df.sample(n=n_samples, random_state=42)

results = []
print('-' * 70)
for i, (_, row) in enumerate(test_samples.iterrows(), 1):
    # strip the task prefix before displaying
    original_msg       = row['input_text'].replace('customer support: ', '')
    actual_response    = row['target_text']
    generated_response = generate_response(original_msg, model, tokenizer, device)

    # measure how close the generated reply is to the real one
    rouge_scores = scorer.score(actual_response, generated_response)

    results.append({
        'Message'  : original_msg[:80],
        'Actual'   : actual_response[:80],
        'Generated': generated_response[:80],
        'ROUGE-L'  : round(rouge_scores['rougeL'].fmeasure, 3)
    })

    print(f'[{i}]')
    print(f'  Message  : {original_msg[:80]}')
    print(f'  Actual   : {actual_response[:80]}')
    print(f'  Generated: {generated_response[:80]}')
    print(f'  ROUGE-L  : {rouge_scores["rougeL"].fmeasure:.3f}')
    print('-' * 70)

results_df = pd.DataFrame(results)
print('Average ROUGE-L:', results_df['ROUGE-L'].mean())

----------------------------------------------------------------------
[1]
  Message  : My package shows delivered but I never received it.
  Actual   : Sorry to hear that. DM us your order number and we will reship it.
  Generated: My package shows delivered but I never received it.
  ROUGE-L  : 0.087
----------------------------------------------------------------------
Average ROUGE-L: 0.087


## Step 14: Save and reload the model

In [20]:
SAVE_PATH = './t5-twitter-customer-service-final'

# save model weights and tokenizer files
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print('Saved to:', SAVE_PATH)

# reload from disk and run a quick test to confirm it works
loaded_tokenizer = T5Tokenizer.from_pretrained(SAVE_PATH)
loaded_model     = T5ForConditionalGeneration.from_pretrained(SAVE_PATH).to(device)

test_msg = "My package has not arrived and it has been 5 days."
response = generate_response(test_msg, loaded_model, loaded_tokenizer, device)
print('Test message:', test_msg)
print('Response    :', response)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to: ./t5-twitter-customer-service-final


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Test message: My package has not arrived and it has been 5 days.
Response    : My package has not arrived and it has been 5 days.


---
## Summary

| Item | Detail |
|------|--------|
| Dataset | Twitter Customer Support (twcs) |
| Model | T5-small / T5-base |
| Task | Seq2Seq: customer message -> support reply |
| Metric | ROUGE-1, ROUGE-2, ROUGE-L |
| Save path | ./t5-twitter-customer-service-final |

### Tips to improve results:
1. Increase epochs from 3 to 10
2. Use t5-base instead of t5-small
3. Use the full dataset (around 3 million tweets)
4. Try a lower learning rate like 1e-4
5. Filter data to keep only high-quality reply pairs